In [5]:
#without GP
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightkurve as lk
import batman
from transitleastsquares import transitleastsquares
import seaborn as sns
import os
from functools import lru_cache
from itertools import product
from scipy.signal import savgol_filter

# ------------------------------------
# 1. Cached light curve loader
# ------------------------------------
@lru_cache(maxsize=10)
def get_lightcurve(tic):
    lc_collection = lk.search_lightcurve(tic, mission="TESS", author="TESS-SPOC", cadence="long").download_all(quality_bitmask="hard")
    if lc_collection is None or len(lc_collection) == 0:
        return None
    
    lc = lc_collection.stitch().remove_nans()



    plt.plot(lc.time.value, lc.flux.value, 'k.', markersize=1, alpha=0.5)
    plt.show()


    return lc.to_pandas().reset_index()

# ------------------------------------
# 2. Flare masking: ±3hr from >5σ events
# ------------------------------------
def mask_flares(df):
    median = df['flux'].median()
    std = df['flux'].std()
    threshold = median + 5 * std
    flare_times = df.loc[df['flux'] > threshold, 'time'].values

    def within_3hr(t):
        return np.any(np.abs(flare_times - t) <= 0.125)

    flare_mask = df['time'].apply(within_3hr)
    return df[~flare_mask].reset_index(drop=True)


# 4. Run TLS detection for a single injection
# ------------------------------------
def run_tls(tic, toi, time, flux, true_period):
    # cadence_days = time[1] - time[0]
    # duration_days = 2
    # wl = int(np.round(duration_days / cadence_days))
    # if wl % 2 == 0:
    #     wl += 1  # make it odd

    # print(wl)
    # flux_filter = savgol_filter(flux, wl, 2)
    # flux = flux / flux_filter

    ##LC PLOT
    # plt.plot(time, flux, 'k.', markersize=1, alpha=0.5)
    # plt.title(f"Lightcurve for {tic}, TOI {toi}")
    # plt.show()

    if len(time) == 0 or len(flux) == 0:
        return False
    tls = transitleastsquares(time, flux)
    result = tls.power(period_max = 25)

    powers = result.power
    periods = result.periods

    periodogram = pd.DataFrame({
        'period': periods,
        'power': powers
    })

    power_7_periods = periodogram[periodogram['power'] > 7]['period'].values
    print(f"TLS Periods with power > 7: {power_7_periods}")
    power_7_power = periodogram[periodogram['power'] > 7]['power'].values


    print(f"{tic}, FAP: {result.FAP}, SDE: {result.SDE}, True Period: {true_period}")
    # Mark as found if period matches true period or a simple alias (1/2x, 2x, 1/3x, 3x, etc.)
    aliases = [true_period, true_period / 2, true_period * 2, true_period / 3, true_period * 3]
    #found = result.FAP < 1e-4 and any(0.99 * alias < result.period < 1.01 * alias for alias in aliases)
    found = any(0.99 * alias < hp < 1.01 * alias for hp in power_7_periods for alias in aliases)


    alias_found = any(0.99 * alias < result.period < 1.01 * alias for alias in aliases)
    print(f"Detection found: {found}")
    if alias_found == True and found == False:
        print(f"True transit or alias found, but high FAP: {alias_found}")
    #print("TLS Transit duration (hours):", result.duration * 24)


    # #TLS phase fold
    # plt.figure()
    # plt.scatter(result.folded_phase, result.folded_y, color='blue', s=10, alpha=0.5,)
    # plt.plot(result.model_folded_phase, result.model_folded_model, color='red')
    # plt.title(f"TLS Phase Folded Lightcurve for {tic}, TOI {toi}")
    # plt.xlabel('Phase')
    # plt.ylabel('Relative flux');
    # plt.show()

    #Periodogram
    plt.figure()
    ax = plt.gca()
    ax.axvline(result.period, alpha=0.4, lw=3)
    plt.xlim(np.min(result.periods), np.max(result.periods))
    for n in range(2, 10):
        ax.axvline(n*result.period, alpha=0.4, lw=1, linestyle="dashed")
        ax.axvline(result.period / n, alpha=0.4, lw=1, linestyle="dashed")
    plt.ylabel(r'SDE')
    plt.xlabel('Period (days)')
    plt.plot(result.periods, result.power, color='black', lw=0.5)
    plt.xlim(0, max(result.periods));
    #plt.xlim(20.8, 21.0)
    plt.show()


    

    return found, alias_found, result.period, result.SDE, power_7_periods, power_7_power, result.period_uncertainty, result.depth, result.rp_rs, result.FAP, result.snr

# ------------------------------------
# 5. Run detection across grid (single process)
# ------------------------------------
def run_detection_for_tic(toi, tic, true_period, true_rp_earth):
    
    print(f"📥 Processing {tic}, toi {toi}")
    lc = get_lightcurve(tic)
    if lc is None:
        print(f"❌ No light curve for {tic}")
        return pd.DataFrame()

    base_df = lc[['time', 'flux']].copy()
    base_df = mask_flares(base_df)

    time = base_df['time']
    flux = base_df['flux']
    

    plt.plot(time, flux, 'k.', markersize=1, alpha=0.5)
    plt.show()

    #flatten lightcurve after flare removal
    lc = lk.LightCurve(time=time, flux=flux)
    time = lc.time.value  # in days
    if len(time) < 2:
        return None

    cadence_days = time[1] - time[0]
    duration_days = 2
    wl = int(np.round(duration_days / cadence_days))
    if wl % 2 == 0:
        wl += 1  # make it odd, as required

    print(f"Cadence: {cadence_days:.5f} days, window_length: {wl}")

    lc_flat = lc.flatten(window_length=wl, polyorder=2)
    flat_df = lc_flat.to_pandas().reset_index()
    time = flat_df['time'].values
    flux = flat_df['flux'].values
    #flux = flux / np.median(flux)

    plt.plot(time, flux, 'k.', markersize=1, alpha=0.5)
    plt.show()


    # Get stellar parameters
    tic_id = int(tic.split()[1])
    star_params = pd.read_csv("TOI_Mar2025_1pt5to4_R_with_extra_columns.csv").query(f"id == {tic_id}")
    if star_params.empty:
        print(f"⚠️ No stellar params for {tic}")
        return pd.DataFrame()

    st_radius = star_params['st_rad'].values[0] #* 6.9634e8
    cdpp = star_params['cdpp_ppm'].values[0]
    st_teff = star_params['st_teff'].values[0]
    st_tmag = star_params['st_tmag'].values[0]

    results = []
    found, alias_found, tls_period, SDE_strongest,  power_array_7, SDE_array_7, tls_period_uncertainty, tls_depth, tls_rp_rs, tls_FAP, tls_snr = run_tls(tic, toi,time, flux, true_period)
    #detections += found

    results.append({
        'TOI': toi,
        'TIC': tic,
        'CDPP (ppm)': round(float(cdpp), 2),
        'Stellar Radius': round(float(st_radius), 2),
        'Stellar Temperature': st_teff,
        'Stellar Magnitude': st_tmag,
        'True Radius (Earth Radii)': round(float(true_rp_earth), 2),
        'True Period (Days)': round(float(true_period), 4),
        'Detection': found,
        #'Alias Detection but high FAP': alias_found,
        'TLS Period': tls_period,
        'TLS SDE strongest': SDE_strongest,
        'TLS SDE > 7 array': SDE_array_7,
        'TLS Periods array': power_array_7,
        'TLS Period Uncertainty': tls_period_uncertainty,
        'TLS Depth': tls_depth,
        'TLS Rp/Rs': tls_rp_rs,
        'TLS FAP': tls_FAP,
        'TLS SNR': tls_snr
    })
        

    return pd.DataFrame(results)

# ------------------------------------
# 6. Save heatmap and results
# ------------------------------------
# def save_heatmap(data, tic):
#     pivot = data.pivot(index="Radius (Earth Radii)", columns="Period (Days)", values="Detection")
#     plt.figure(figsize=(8, 6))
#     sns.heatmap(pivot, cmap="viridis", vmin=0, vmax=1, cbar_kws={'label': 'Percent found'}).invert_yaxis()
#     plt.title(f"{tic} noise transit detection rate")
#     plt.xlabel("Period (Days)")
#     plt.ylabel("Radius (Earth Radii)")
#     plt.tight_layout()
#     plt.show()

#     os.makedirs("heatmap_data", exist_ok=True)
#     path = f'heatmap_data/{tic.replace(" ", "_")}_cel_heatmap.csv'
#     data.to_csv(path, index=False)
#     print(f"✅ Saved: {path}")

# ------------------------------------
# 7. Main driver
# ------------------------------------
if __name__ == "__main__":

    save_file = "./rec_results/all_results_to_25P_multi_savgolay_2dayfilter_overSDE7_flares_first.csv"
    #save_file = './test.csv'

    # Load TIC IDs from the CSV file
    star_params = pd.read_csv("TOI_Mar2025_1pt5to4_R_with_Mst.csv")
    #star_params = star_params.iloc[530:].reset_index(drop=True)
    # Filter for TOI == 7033
    #star_params = star_params[star_params["toi"] == 7033.01]
    #star_params = star_params[928:]
    star_params = star_params[star_params["pl_orbper"] <= 25]
    print(len(star_params))
    
        # Load existing results to check for already processed TOIs
    if os.path.exists(save_file):
        existing_results = pd.read_csv(save_file)
        processed_tois = set(existing_results[existing_results['Stellar Radius'].notna()]['TOI'].values)
        print(f"Found {len(processed_tois)} already processed TOIs with valid results")
    else:
        processed_tois = set()
        print("No existing results file found, starting fresh")

    # Then modify the main loop to check for already processed TOIs:
for _, row in star_params.iterrows():
    toi = row['toi']
    tic = f"TIC {int(row['id'])}"
    true_period = row['pl_orbper']
    true_rp_earth = row['pl_rade']
    
    if toi in processed_tois:
        print(f"⏭️ Skipping TOI {toi} - already processed")
        continue

    df_save = pd.DataFrame()
    
    try:
        df_results = run_detection_for_tic(toi, tic, true_period, true_rp_earth)

        if not df_results.empty:
            df_save = pd.concat([df_save, df_results], ignore_index=True)
        else:
            df_results = pd.DataFrame([{
                'TOI': toi,
                'TIC': tic,
                'CDPP (ppm)': None,
                'Stellar Radius': None,
                'Stellar Temperature': None,
                'Stellar Magnitude': None,
                'True Radius (Earth Radii)': None,
                'True Period (Days)': None,
                'Detection': "Empty",
                'Alias Detection but high FAP': None,
                'TLS Period': None,
                'TLS SDE strongest': None,
                'TLS SDE > 7 array': None,
                'TLS Periods array': None,
                'TLS Period Uncertainty': None,
                'TLS Depth': None,
                'TLS Rp/Rs': None,
                'TLS FAP': None,
                'TLS SNR': None
            }])
            df_save = pd.concat([df_save, df_results], ignore_index=True)

        # ✅ Save result
        if os.path.exists(save_file):
            existing = pd.read_csv(save_file)
            existing = existing[existing['TOI'] != toi]
            updated = pd.concat([existing, df_results], ignore_index=True)
        else:
            updated = df_results
        updated.to_csv(save_file, index=False)

    except Exception as e:
        print(f"❌ Error occurred for {tic}: {e}")
        df_results = pd.DataFrame([{
            'TOI': toi,
            'TIC': tic,
            'CDPP (ppm)': None,
            'Stellar Radius': None,
            'Stellar Temperature': None,
            'Stellar Magnitude': None,
            'True Radius (Earth Radii)': None,
            'True Period (Days)': None,
            'Detection': "Error",
            'Alias Detection but high FAP': None,
            'TLS Period': None,
            'TLS SDE strongest': None,
            'TLS SDE > 7 array': None,
            'TLS Periods array': None,
            'TLS Period Uncertainty': None,
            'TLS Depth': None,
            'TLS Rp/Rs': None,
            'TLS FAP': None,
            'TLS SNR': None
        }])
        df_save = pd.concat([df_save, df_results], ignore_index=True)

        # ✅ Save even on error
        if os.path.exists(save_file):
            existing = pd.read_csv(save_file)
            existing = existing[existing['TOI'] != toi]
            updated = pd.concat([existing, df_results], ignore_index=True)
        else:
            updated = df_results
        updated.to_csv(save_file, index=False)

1036
Found 995 already processed TOIs with valid results
⏭️ Skipping TOI 5165.01 - already processed
⏭️ Skipping TOI 512.01 - already processed
⏭️ Skipping TOI 7043.01 - already processed
📥 Processing TIC 63100069, toi 7170.01
❌ No light curve for TIC 63100069


/Users/danayaptangco/opt/anaconda3/envs/Mulders/lib/python3.9/site-packages/lightkurve/search.py:494: LightkurveWarning: Cannot download from an empty search result.
  warnings.warn(


⏭️ Skipping TOI 6940.01 - already processed
⏭️ Skipping TOI 6944.01 - already processed
⏭️ Skipping TOI 5789.01 - already processed
⏭️ Skipping TOI 7033.01 - already processed
⏭️ Skipping TOI 5319.01 - already processed
⏭️ Skipping TOI 5082.01 - already processed
⏭️ Skipping TOI 6054.01 - already processed
⏭️ Skipping TOI 6054.02 - already processed
📥 Processing TIC 96475638, toi 6657.01
❌ No light curve for TIC 96475638


/Users/danayaptangco/opt/anaconda3/envs/Mulders/lib/python3.9/site-packages/lightkurve/search.py:494: LightkurveWarning: Cannot download from an empty search result.
  warnings.warn(


⏭️ Skipping TOI 6699.01 - already processed
⏭️ Skipping TOI 6826.01 - already processed
⏭️ Skipping TOI 6858.01 - already processed
⏭️ Skipping TOI 6904.01 - already processed
⏭️ Skipping TOI 6911.01 - already processed
⏭️ Skipping TOI 5159.01 - already processed
📥 Processing TIC 47319867, toi 5423.02
❌ No light curve for TIC 47319867
📥 Processing TIC 51768374, toi 7177.01
❌ No light curve for TIC 51768374


...

/Users/danayaptangco/opt/anaconda3/envs/Mulders/lib/python3.9/site-packages/lightkurve/search.py:494: LightkurveWarning: Cannot download from an empty search result.
  warnings.warn(
/Users/danayaptangco/opt/anaconda3/envs/Mulders/lib/python3.9/site-packages/lightkurve/search.py:494: LightkurveWarning: Cannot download from an empty search result.
  warnings.warn(


⏭️ Skipping TOI 1438.01 - already processed
⏭️ Skipping TOI 1438.02 - already processed
⏭️ Skipping TOI 1440.01 - already processed
⏭️ Skipping TOI 1716.01 - already processed
⏭️ Skipping TOI 2076.03 - already processed
⏭️ Skipping TOI 2494.02 - already processed
⏭️ Skipping TOI 1444.01 - already processed
⏭️ Skipping TOI 1437.01 - already processed
⏭️ Skipping TOI 1737.01 - already processed
📥 Processing TIC 234832821, toi 2590.01
❌ No light curve for TIC 234832821
⏭️ Skipping TOI 1448.01 - already processed
⏭️ Skipping TOI 280.01 - already processed
⏭️ Skipping TOI 1117.01 - already processed
⏭️ Skipping TOI 3355.01 - already processed
⏭️ Skipping TOI 1602.01 - already processed
⏭️ Skipping TOI 5744.01 - already processed
⏭️ Skipping TOI 1473.01 - already processed
⏭️ Skipping TOI 1537.01 - already processed
⏭️ Skipping TOI 1293.01 - already processed
⏭️ Skipping TOI 2160.01 - already processed
⏭️ Skipping TOI 5988.01 - already processed
⏭️ Skipping TOI 5994.01 - already processed
⏭️

/Users/danayaptangco/opt/anaconda3/envs/Mulders/lib/python3.9/site-packages/lightkurve/search.py:494: LightkurveWarning: Cannot download from an empty search result.
  warnings.warn(
/Users/danayaptangco/opt/anaconda3/envs/Mulders/lib/python3.9/site-packages/lightkurve/search.py:494: LightkurveWarning: Cannot download from an empty search result.
  warnings.warn(


⏭️ Skipping TOI 2495.01 - already processed
📥 Processing TIC 393298446, toi 2506.01
❌ No light curve for TIC 393298446
⏭️ Skipping TOI 2507.01 - already processed
⏭️ Skipping TOI 2512.01 - already processed
⏭️ Skipping TOI 2513.01 - already processed
⏭️ Skipping TOI 2519.01 - already processed
⏭️ Skipping TOI 2523.01 - already processed
⏭️ Skipping TOI 256.01 - already processed
⏭️ Skipping TOI 261.01 - already processed
⏭️ Skipping TOI 270.02 - already processed
⏭️ Skipping TOI 271.01 - already processed
⏭️ Skipping TOI 279.01 - already processed
⏭️ Skipping TOI 314.01 - already processed
⏭️ Skipping TOI 396.02 - already processed
⏭️ Skipping TOI 4183.01 - already processed
⏭️ Skipping TOI 4190.01 - already processed
⏭️ Skipping TOI 4296.01 - already processed
⏭️ Skipping TOI 4297.01 - already processed
⏭️ Skipping TOI 4311.01 - already processed
⏭️ Skipping TOI 4352.01 - already processed
⏭️ Skipping TOI 4354.01 - already processed
⏭️ Skipping TOI 4363.01 - already processed
⏭️ Skipp

/Users/danayaptangco/opt/anaconda3/envs/Mulders/lib/python3.9/site-packages/lightkurve/search.py:494: LightkurveWarning: Cannot download from an empty search result.
  warnings.warn(
